# KuaiRand ALS Baseline

This notebook reports the collaborative-filtering baseline artifacts generated by reusable code in `recommender/gold/als_baseline.py`.

Run the pipeline from the repository root:

```bash
python scripts/run_kuairand_als_baseline.py --output-dir data/gold/als/v1 --overwrite
```


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'recommender').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import Window
from pyspark.sql import functions as F
from recommender.spark import get_spark

spark = get_spark('kuairand-als-baseline-report', reset=True)
spark.sparkContext.setLogLevel('WARN')

GOLD_DIR = PROJECT_ROOT / 'data/gold/als/v1'
MANIFEST_PATH = GOLD_DIR / 'manifest.json'
if not MANIFEST_PATH.exists():
    raise FileNotFoundError('Missing manifest. Run scripts/run_kuairand_als_baseline.py first.')
manifest = json.loads(MANIFEST_PATH.read_text())
print(f'Gold dir: {GOLD_DIR}')
print(f"Generated at: {manifest['generated_at']}")
print(f"Code commit: {manifest.get('code_commit')}")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/23 21:34:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/23 21:34:19 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


Gold dir: /Users/khoatran/coding/recsys/data/gold/als/v1
Generated at: 2026-09-24T02:32:04.946245+00:00
Code commit: cabc9e106b0dccdf4050b064712812ea2b62ec19


## Temporal Split and Interaction Formula

The split is event-level chronological before user-item aggregation. Interaction strength is non-negative and excludes negative feedback in V1.


In [2]:
print('Temporal cutoffs')
print(json.dumps(manifest['temporal_split'], indent=2))
print('\nInteraction strength')
print(json.dumps(manifest['interaction_strength'], indent=2))
spark.createDataFrame(manifest['split_summary']).select('split','events','users','items','min_ts','max_ts').orderBy('split').show(truncate=False)


Temporal cutoffs
{
  "train_cutoff_time_ms": 1651305433084,
  "train_cutoff_ts": "2022-04-30T02:57:13",
  "validation_cutoff_time_ms": 1651654220144,
  "validation_cutoff_ts": "2022-05-04T03:50:20"
}

Interaction strength
{
  "formula": "log1p(sum(0.5*clip(watch_ratio,0,3)+1.0*long_view+1.5*like+1.5*comment+1.5*forward+2.0*follow)) with selected weights if tuning is enabled",
  "formula_version": "implicit_strength_v1",
  "negative_feedback_policy": "is_hate is retained in silver but not encoded as negative ALS rating in v1",
  "weights": {
    "is_comment": 1.5,
    "is_follow": 2.0,
    "is_forward": 1.5,
    "is_like": 1.5,
    "long_view": 1.0,
    "watch_ratio": 0.5
  }
}


+----------+-------+-----+-------+-------------------+-------------------+
|split     |events |users|items  |min_ts             |max_ts             |
+----------+-------+-----+-------+-------------------+-------------------+
|test      |1766483|986  |885555 |2022-05-04T03:50:20|2022-05-08T10:52:09|
|train     |8229365|994  |3218299|2022-04-07T09:00:03|2022-04-30T02:57:10|
|validation|1760225|990  |852144 |2022-04-30T02:57:13|2022-05-04T03:50:18|
+----------+-------+-----+-------+-------------------+-------------------+



## Gold Dataset Statistics


In [3]:
train = spark.read.parquet(str(GOLD_DIR / 'train_interactions'))
val_rel = spark.read.parquet(str(GOLD_DIR / 'validation_relevance'))
test_rel = spark.read.parquet(str(GOLD_DIR / 'test_relevance'))
users = spark.read.parquet(str(GOLD_DIR / 'user_mapping'))
items = spark.read.parquet(str(GOLD_DIR / 'item_mapping'))
user_factors = spark.read.parquet(str(GOLD_DIR / 'user_factors'))
item_factors = spark.read.parquet(str(GOLD_DIR / 'item_factors'))

for name, df in [('train_interactions', train), ('validation_relevance', val_rel), ('test_relevance', test_rel), ('user_mapping', users), ('item_mapping', items), ('user_factors', user_factors), ('item_factors', item_factors)]:
    print(f'{name}: rows={df.count():,}, columns={len(df.columns)}')
    print(df.columns)


train_interactions: rows=6,111,386, columns=5
['user_idx', 'video_idx', 'interaction_strength', 'user_id', 'video_id']
validation_relevance: rows=473,597, columns=9
['video_id', 'user_id', 'relevant', 'graded_relevance', 'relevant_events', 'user_idx', 'video_idx', 'is_warm_user', 'is_warm_item']
test_relevance: rows=466,417, columns=9
['video_id', 'user_id', 'relevant', 'graded_relevance', 'relevant_events', 'user_idx', 'video_idx', 'is_warm_user', 'is_warm_item']
user_mapping: rows=994, columns=2
['user_id', 'user_idx']


item_mapping: rows=2,225,231, columns=2
['video_id', 'video_idx']
user_factors: rows=999, columns=2
['user_idx', 'features']
item_factors: rows=2,612,718, columns=2
['video_idx', 'features']


## Popularity vs ALS Metrics

Metrics are ranking metrics at K=10. Recommendations exclude train-history items. Cold-start entities are reported separately instead of silently disappearing from the dataset report.


In [4]:
rows = []
for model_name, section in [('popularity', manifest['popularity_baseline']), ('als_final', {'test': manifest['final_als_test_metrics']})]:
    for split, metrics in section.items():
        row = {'model': model_name, 'split': split}
        row.update(metrics)
        rows.append(row)
metrics_df = spark.createDataFrame(rows)
metrics_df.show(truncate=False)

print('Selected ALS hyperparameters')
print(json.dumps(manifest['selected_als_hyperparameters'], indent=2))
print('\nCold-start report')
print(json.dumps(manifest['cold_start'], indent=2))


+---------------------+---------------+-------------------+----------+--------------------+--------------------+----------+
|avg_recommended_items|evaluated_users|hitrate@10         |model     |ndcg@10             |recall@10           |split     |
+---------------------+---------------+-------------------+----------+--------------------+--------------------+----------+
|10.0                 |973            |0.04316546762589928|popularity|0.002872687228335957|0.004419321685508737|test      |
|10.0                 |978            |0.07668711656441718|popularity|0.006595566404320446|0.008486707566462168|validation|
|10.0                 |980            |0.1336734693877551 |als_final |0.009890709882088127|0.015510204081632655|test      |
+---------------------+---------------+-------------------+----------+--------------------+--------------------+----------+

Selected ALS hyperparameters
{
  "alpha": 10.0,
  "max_iter": 5,
  "rank": 32,
  "reg_param": 0.05
}

Cold-start report
{
  "final_

## ALS Grid Results


In [5]:
grid_rows = []
for result in manifest['als_grid_results']:
    row = dict(result['params'])
    row.update(result['validation'])
    grid_rows.append(row)
spark.createDataFrame(grid_rows).orderBy(F.desc('ndcg@10')).show(truncate=False)


+-----+---------------------+---------------+-------------------+--------+--------------------+----+-------------------+---------+
|alpha|avg_recommended_items|evaluated_users|hitrate@10         |max_iter|ndcg@10             |rank|recall@10          |reg_param|
+-----+---------------------+---------------+-------------------+--------+--------------------+----+-------------------+---------+
|10.0 |10.0                 |978            |0.19120654396728015|5       |0.014338878714535177|32  |0.02280163599182005|0.05     |
+-----+---------------------+---------------+-------------------+--------+--------------------+----+-------------------+---------+



## Recommendation Examples From Saved Factors

This computes dot products from saved user/item latent vectors for a few warm test users and filters train-history items.


In [6]:
example_users = test_rel.where(F.col('is_warm_user')).select('user_idx', 'user_id').distinct().orderBy('user_idx').limit(3)
example_users.show()

u = user_factors.join(example_users.select('user_idx'), 'user_idx').select('user_idx', F.col('features').alias('user_features'))
i = item_factors.select('video_idx', F.col('features').alias('item_features'))
history = train.select('user_idx', 'video_idx')
item_ids = items.select('video_idx', 'video_id')

scored = (
    u.crossJoin(i)
    .withColumn('score', F.expr('aggregate(zip_with(user_features, item_features, (x, y) -> x * y), cast(0.0 as double), (acc, x) -> acc + x)'))
    .join(history, ['user_idx', 'video_idx'], 'left_anti')
)
ranked = scored.withColumn('rank', F.row_number().over(Window.partitionBy('user_idx').orderBy(F.desc('score'), F.asc('video_idx')))).where(F.col('rank') <= 10)
ranked.join(example_users, 'user_idx').join(item_ids, 'video_idx').select('user_id', 'video_id', 'rank', 'score').orderBy('user_id', 'rank').show(50, truncate=False)


+--------+-------+
|user_idx|user_id|
+--------+-------+
|       0|      0|
|       1|      1|
|       2|      2|
+--------+-------+



+-------+--------+----+-------------------+
|user_id|video_id|rank|score              |
+-------+--------+----+-------------------+
|0      |3565656 |1   |0.5600688835402252 |
|0      |605904  |2   |0.5048104571906151 |
|0      |684344  |3   |0.49981826639850624|
|0      |295304  |4   |0.4995305099655525 |
|0      |2695044 |5   |0.4945916512515396 |
|0      |510144  |6   |0.4871501014131354 |
|0      |151132  |7   |0.4866553679239587 |
|0      |2568345 |8   |0.48594376508845016|
|0      |1101611 |9   |0.48343261462287046|
|0      |3210624 |10  |0.4824304538051365 |
|1      |660810  |1   |0.41060131933772936|
|1      |2906493 |2   |0.40648350933770416|
|1      |3459505 |3   |0.4027003715891624 |
|1      |443814  |4   |0.4002059002232272 |
|1      |4240615 |5   |0.3979975610627662 |
|1      |3162374 |6   |0.3957324969378533 |
|1      |2977295 |7   |0.3956564831605647 |
|1      |2585463 |8   |0.3933739615599734 |
|1      |2563868 |9   |0.3922298394008976 |
|1      |2997666 |10  |0.3913900

## Limitations

- This is an implicit-feedback collaborative-filtering baseline, not the final adaptive ranker.
- Cold users/items are not solved by ALS; they are measured and should be handled later with content/session features.
- `videos_statistics` aggregate rates are excluded from ALS strength because their snapshot time is unknown.
- `is_hate` is retained in silver but not encoded as negative ALS feedback in V1.
- The local executed run used a compact ALS grid; broaden the grid when compute is available.
